# 16. Global Fragility Scenarios

This notebook adds a presentation-ready sensitivity layer to the current Global Fragility Index. The historical weights remain fixed while the present-day scoring assumptions are varied across three editable scenarios.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("default")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

## Load Current Global Scoring Template

In [ ]:
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

template_path = project_root / "data" / "processed" / "global_fragility_current_scoring_template.csv"
scoring_df = pd.read_csv(template_path)

scoring_df["confidence_level"] = scoring_df["confidence_level"].fillna("")
scoring_df["evidence_notes"] = scoring_df["evidence_notes"].fillna("")
scoring_df["baseline_input_score"] = pd.to_numeric(scoring_df["current_global_score_0_to_3"], errors="coerce")

role_map = {factor_name: "stress" for factor_name in scoring_df["factor_name"]}
role_map["adaptive_capacity_resilience"] = "resilience"
scoring_df["role"] = scoring_df["factor_name"].map(role_map)

display(scoring_df[[
    "factor_name",
    "historical_category",
    "historical_weight",
    "baseline_input_score",
    "confidence_level",
    "role",
]])

## Scenario Defaults and Variability Flags

The earlier scenario notebook produced overly mechanical differences because it applied a blanket `+/-1` rule across all factors and then used those values directly.

This revised version separates two steps:
1. **Provisional automatic defaults** created from the blanket `+/-1` rule
2. **Editable active scenario scores** used in the calculations

A suggested variability flag is also added for each factor:
- `fixed_across_scenarios`
- `moderately_variable`
- `highly_scenario_sensitive`

These flags are just first-pass editing aids. They do not change the historical weights and they can be revised factor by factor.

In [ ]:
scenario_sheet = scoring_df[[
    "factor_name",
    "historical_category",
    "historical_weight",
    "role",
    "confidence_level",
    "evidence_notes",
    "baseline_input_score",
]].copy()

def auto_default_score(row, scenario_name):
    baseline = row["baseline_input_score"]
    if pd.isna(baseline):
        return pd.NA
    baseline = int(baseline)
    is_resilience = row["role"] == "resilience"
    if scenario_name == "baseline":
        return baseline
    if scenario_name == "conservative":
        return min(baseline + 1, 3) if is_resilience else max(baseline - 1, 0)
    if scenario_name == "high_stress":
        return max(baseline - 1, 0) if is_resilience else min(baseline + 1, 3)
    raise KeyError(f"Unknown scenario_name: {scenario_name}")

scenario_sheet["auto_conservative_default_0_to_3"] = scenario_sheet.apply(auto_default_score, axis=1, scenario_name="conservative")
scenario_sheet["auto_baseline_default_0_to_3"] = scenario_sheet.apply(auto_default_score, axis=1, scenario_name="baseline")
scenario_sheet["auto_high_stress_default_0_to_3"] = scenario_sheet.apply(auto_default_score, axis=1, scenario_name="high_stress")

def suggested_variability_flag(row):
    confidence = str(row["confidence_level"]).strip().lower()
    if confidence == "high":
        return "fixed_across_scenarios"
    if confidence == "low":
        return "highly_scenario_sensitive"
    return "moderately_variable"

scenario_sheet["scenario_variability_flag"] = scenario_sheet.apply(suggested_variability_flag, axis=1)

variability_note_map = {
    "fixed_across_scenarios": "Suggested to stay stable across scenarios unless you have a strong reason to vary it.",
    "moderately_variable": "Suggested to vary somewhat across scenarios, but not wildly.",
    "highly_scenario_sensitive": "Suggested to receive closer judgment-based refinement because baseline certainty is weaker.",
}
scenario_sheet["scenario_variability_note"] = scenario_sheet["scenario_variability_flag"].map(variability_note_map)

scenario_sheet["default_generation_note"] = "Provisional defaults generated from the blanket +/-1 rule; editable below."
scenario_sheet["manual_edit_note"] = ""

scenario_sheet["conservative_score_0_to_3"] = scenario_sheet["auto_conservative_default_0_to_3"]
scenario_sheet["baseline_score_0_to_3"] = scenario_sheet["auto_baseline_default_0_to_3"]
scenario_sheet["high_stress_score_0_to_3"] = scenario_sheet["auto_high_stress_default_0_to_3"]

fixed_mask = scenario_sheet["scenario_variability_flag"].eq("fixed_across_scenarios")
scenario_sheet.loc[fixed_mask, "conservative_score_0_to_3"] = scenario_sheet.loc[fixed_mask, "baseline_input_score"]
scenario_sheet.loc[fixed_mask, "baseline_score_0_to_3"] = scenario_sheet.loc[fixed_mask, "baseline_input_score"]
scenario_sheet.loc[fixed_mask, "high_stress_score_0_to_3"] = scenario_sheet.loc[fixed_mask, "baseline_input_score"]

display(scenario_sheet[[
    "factor_name",
    "role",
    "confidence_level",
    "scenario_variability_flag",
    "auto_conservative_default_0_to_3",
    "auto_baseline_default_0_to_3",
    "auto_high_stress_default_0_to_3",
    "conservative_score_0_to_3",
    "baseline_score_0_to_3",
    "high_stress_score_0_to_3",
]])

## Manual Scenario Refinement

Use this section to refine the active scenario values factor by factor.

- The **auto default** columns are preserved for reference.
- The **active scenario score** columns are what the notebook actually uses.
- For factors marked `fixed_across_scenarios`, the active scores are initially aligned to the baseline score.
- For factors marked `moderately_variable` or `highly_scenario_sensitive`, the active scores start from the automatic defaults.

The simplest way to edit a factor is to add it to the `manual_factor_overrides` dictionary below.

In [ ]:
manual_factor_overrides = {
    # "legitimacy_crisis": {
    #     "scenario_variability_flag": "moderately_variable",
    #     "conservative_score_0_to_3": 1,
    #     "baseline_score_0_to_3": 2,
    #     "high_stress_score_0_to_3": 3,
    #     "manual_edit_note": "Example override.",
    # },
}

editable_columns = {
    "scenario_variability_flag",
    "scenario_variability_note",
    "conservative_score_0_to_3",
    "baseline_score_0_to_3",
    "high_stress_score_0_to_3",
    "manual_edit_note",
}

for factor_name, updates in manual_factor_overrides.items():
    row_mask = scenario_sheet["factor_name"].eq(factor_name)
    if not row_mask.any():
        raise KeyError(f"Unknown factor_name in manual_factor_overrides: {factor_name}")
    for key, value in updates.items():
        if key not in editable_columns:
            raise KeyError(f"Unsupported editable column: {key}")
        scenario_sheet.loc[row_mask, key] = value

display(scenario_sheet[[
    "factor_name",
    "historical_category",
    "role",
    "confidence_level",
    "scenario_variability_flag",
    "scenario_variability_note",
    "auto_conservative_default_0_to_3",
    "auto_baseline_default_0_to_3",
    "auto_high_stress_default_0_to_3",
    "conservative_score_0_to_3",
    "baseline_score_0_to_3",
    "high_stress_score_0_to_3",
    "manual_edit_note",
]])

## Why Blanket `+/-1` Is Only a Starting Point

A blanket rule is useful for generating quick scenario variation, but it is too coarse to stand on its own.

- Some factors should remain fairly stable across scenarios because the present evidence is broad and robust.
- Some factors are more judgment-sensitive because the evidence is mixed, the baseline score is less secure, or the factor is harder to translate into a single global value.
- The scenario table above is therefore designed for manual refinement, not automatic final truth claims.

In practice, it is often more defensible to keep clearly evidenced global stresses stable across scenarios and vary only the more uncertain or interpretation-sensitive factors.

In [ ]:
variability_summary = (
    scenario_sheet.groupby("scenario_variability_flag")
    .agg(
        factor_count=("factor_name", "count"),
        average_weight=("historical_weight", "mean"),
    )
    .reset_index()
    .sort_values("factor_count", ascending=False)
)

display(variability_summary)
display(scenario_sheet[["factor_name", "scenario_variability_flag", "scenario_variability_note"]])

## Scenario Calculations

In [ ]:
scenario_long = scenario_sheet.melt(
    id_vars=[
        "factor_name",
        "historical_category",
        "historical_weight",
        "role",
        "confidence_level",
        "evidence_notes",
        "scenario_variability_flag",
        "manual_edit_note",
    ],
    value_vars=[
        "conservative_score_0_to_3",
        "baseline_score_0_to_3",
        "high_stress_score_0_to_3",
    ],
    var_name="scenario_column",
    value_name="scenario_score",
)

scenario_long["scenario"] = scenario_long["scenario_column"].map({
    "conservative_score_0_to_3": "conservative",
    "baseline_score_0_to_3": "baseline",
    "high_stress_score_0_to_3": "high_stress",
})
scenario_long["score_numeric"] = pd.to_numeric(scenario_long["scenario_score"], errors="coerce")

invalid_mask = scenario_long["score_numeric"].notna() & ~scenario_long["score_numeric"].isin([0, 1, 2, 3])
if invalid_mask.any():
    invalid_rows = scenario_long.loc[invalid_mask, ["scenario", "factor_name", "scenario_score"]]
    raise ValueError(f"Scenario scores must be 0, 1, 2, or 3. Invalid rows:\n{invalid_rows}")

scenario_long["weighted_score"] = scenario_long["historical_weight"] * (scenario_long["score_numeric"] / 3)
scenario_long["stress_contribution"] = 0.0
scenario_long["resilience_support"] = 0.0
scenario_long["net_fragility_contribution"] = pd.NA

stress_rows = scenario_long["role"].eq("stress") & scenario_long["score_numeric"].notna()
resilience_rows = scenario_long["role"].eq("resilience") & scenario_long["score_numeric"].notna()

scenario_long.loc[stress_rows, "stress_contribution"] = scenario_long.loc[stress_rows, "weighted_score"]
scenario_long.loc[stress_rows, "net_fragility_contribution"] = scenario_long.loc[stress_rows, "weighted_score"]

scenario_long.loc[resilience_rows, "resilience_support"] = scenario_long.loc[resilience_rows, "weighted_score"]
scenario_long.loc[resilience_rows, "net_fragility_contribution"] = (
    scenario_long.loc[resilience_rows, "historical_weight"] * (1 - (scenario_long.loc[resilience_rows, "score_numeric"] / 3))
)

display(scenario_long[[
    "scenario",
    "factor_name",
    "role",
    "scenario_variability_flag",
    "score_numeric",
    "historical_weight",
    "weighted_score",
    "net_fragility_contribution",
]])

## Scenario Summary

For each scenario, the notebook computes:
- `weighted_stress_score`
- `resilience_adjustment`
- `net_fragility_score`
- `normalized_0_to_100_score`

Interpretation bands:
- `0–20` = low overlap
- `21–40` = mild fragility
- `41–60` = moderate fragility
- `61–80` = high fragility
- `81–100` = severe systemic stress

In [ ]:
stress_weight_total = scenario_long.loc[scenario_long["role"].eq("stress"), "historical_weight"].drop_duplicates().sum()
resilience_weight_total = scenario_long.loc[scenario_long["role"].eq("resilience"), "historical_weight"].drop_duplicates().sum()

def interpret_band(value):
    if pd.isna(value):
        return "No score yet"
    if value <= 20:
        return "Low overlap"
    if value <= 40:
        return "Mild fragility"
    if value <= 60:
        return "Moderate fragility"
    if value <= 80:
        return "High fragility"
    return "Severe systemic stress"

scenario_summary_rows = []
for scenario_name, group in scenario_long.groupby("scenario", sort=False):
    entered_mask = group["score_numeric"].notna()
    stress_mask = group["role"].eq("stress") & entered_mask
    resilience_mask = group["role"].eq("resilience") & entered_mask
    entered_weight_total = group.loc[entered_mask, "historical_weight"].sum()

    weighted_stress_score = (
        100 * group.loc[stress_mask, "weighted_score"].sum() / stress_weight_total
        if stress_mask.any() else pd.NA
    )
    resilience_adjustment = (
        100 * group.loc[resilience_mask, "weighted_score"].sum() / resilience_weight_total
        if resilience_mask.any() else pd.NA
    )
    net_fragility_score = (
        100 * group.loc[entered_mask, "net_fragility_contribution"].sum() / entered_weight_total
        if entered_mask.any() else pd.NA
    )

    scenario_summary_rows.append({
        "scenario": scenario_name,
        "entered_factor_count": int(entered_mask.sum()),
        "weighted_stress_score": weighted_stress_score,
        "resilience_adjustment": resilience_adjustment,
        "net_fragility_score": net_fragility_score,
        "normalized_0_to_100_score": net_fragility_score,
        "interpretation_band": interpret_band(net_fragility_score),
    })

scenario_summary = pd.DataFrame(scenario_summary_rows)
scenario_order = ["conservative", "baseline", "high_stress"]
scenario_summary["scenario"] = pd.Categorical(scenario_summary["scenario"], categories=scenario_order, ordered=True)
scenario_summary = scenario_summary.sort_values("scenario").reset_index(drop=True)

display(scenario_summary)

## Scenario Comparison Chart

In [ ]:
plot_df = scenario_summary.copy()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(plot_df["scenario"].astype(str), plot_df["normalized_0_to_100_score"], color=["steelblue", "darkorange", "firebrick"])
ax.set_ylim(0, 100)
ax.set_ylabel("Normalized 0-100 fragility score")
ax.set_title("Global Fragility Scenario Comparison")
for bar, value in zip(bars, plot_df["normalized_0_to_100_score"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 1.5, f"{value:.1f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## Stress Drivers vs Resilience / Stabilizing Contributions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(plot_df))
width = 0.36

ax.bar([i - width / 2 for i in x], plot_df["weighted_stress_score"], width=width, label="Stress-driver score", color="indianred")
ax.bar([i + width / 2 for i in x], plot_df["resilience_adjustment"], width=width, label="Resilience / stabilizing score", color="seagreen")
ax.set_xticks(list(x))
ax.set_xticklabels(plot_df["scenario"].astype(str))
ax.set_ylim(0, 100)
ax.set_ylabel("Normalized score")
ax.set_title("Stress Drivers vs Resilience By Scenario")
ax.legend()
plt.tight_layout()
plt.show()

## Factor-Level Scenario Contributions

This table is the main editing surface for report writing and scenario refinement. It shows the active score, the factor’s variability flag, and the weighted contribution in each scenario.

In [ ]:
factor_contribution_table = scenario_long[[
    "scenario",
    "factor_name",
    "historical_category",
    "role",
    "scenario_variability_flag",
    "score_numeric",
    "historical_weight",
    "weighted_score",
    "confidence_level",
    "manual_edit_note",
]].sort_values(["scenario", "weighted_score"], ascending=[True, False])

display(factor_contribution_table)

## Reporting Note

Use this notebook to show how much the present-day overlap score changes under different reasonable judgment calls while keeping the underlying historical framework constant.
